# 📊 01 — Data Overview & EDA Readiness

**Project:** AI Smart Inventory Management & Demand Forecasting System  
**Phase:** Phase 3 — Data Extraction, Cleaning & Pipeline Engineering  
**Day:** Day 7 (EDA Readiness)  
**Prepared by:** Antigravity  
**Date:** 2026-07-07  

---

## Objectives

This notebook:
1. Loads all processed CSVs from `data/processed/`
2. Displays schemas and data types
3. Shows descriptive statistics per table
4. Verifies completeness, uniqueness, and value ranges
5. Confirms the dataset is ready for Phase 4 (EDA)

---

In [ ]:
import os
import pandas as pd
import numpy as np

# Suppress display truncation
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.width', 200)

print('✅ Libraries loaded successfully')

## 1. Load Processed Datasets

In [ ]:
# Resolve paths relative to this notebook
NOTEBOOK_DIR = os.path.abspath('')
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', '..'))
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

print(f'Project Root  : {PROJECT_ROOT}')
print(f'Processed Dir : {PROCESSED_DIR}')

# List available processed files
processed_files = [f for f in os.listdir(PROCESSED_DIR) if f.endswith('_processed.csv')]
print(f'\nFound {len(processed_files)} processed CSV files:')
for f in sorted(processed_files):
    fpath = os.path.join(PROCESSED_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f:<40} ({size_kb:.1f} KB)')

In [ ]:
# Load all processed tables
tables = {}
for fname in sorted(processed_files):
    table_name = fname.replace('_processed.csv', '')
    fpath = os.path.join(PROCESSED_DIR, fname)
    tables[table_name] = pd.read_csv(fpath, low_memory=False)
    print(f'  Loaded [{table_name}]: {tables[table_name].shape[0]:,} rows × {tables[table_name].shape[1]} columns')

print(f'\n✅ All {len(tables)} tables loaded successfully')

## 2. Dataset Summary (Schemas & Row Counts)

In [ ]:
# Summary table
summary_rows = []
for name, df in tables.items():
    null_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1]) * 100)
    numeric_cols = df.select_dtypes(include='number').shape[1]
    summary_rows.append({
        'Table': name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'Numeric Cols': numeric_cols,
        'Non-Numeric Cols': df.shape[1] - numeric_cols,
        'Null %': round(null_pct, 2)
    })

summary_df = pd.DataFrame(summary_rows)
print('=== DATASET SUMMARY ===')
print(summary_df.to_string(index=False))

## 3. Schema Inspection (Column Types)

In [ ]:
for name, df in tables.items():
    print(f'\n{'='*60}')
    print(f'TABLE: {name.upper()} ({df.shape[0]:,} rows × {df.shape[1]} cols)')
    print(f'{'='*60}')
    dtype_info = pd.DataFrame({
        'Column': df.columns,
        'Dtype': df.dtypes.values,
        'Non-Null': df.notnull().sum().values,
        'Null %': (df.isnull().mean() * 100).round(2).values,
        'Sample': [str(df[c].dropna().iloc[0]) if not df[c].dropna().empty else 'N/A' for c in df.columns]
    })
    print(dtype_info.to_string(index=False))

## 4. Descriptive Statistics

In [ ]:
for name, df in tables.items():
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.empty:
        print(f'[{name}]: No numeric columns to describe.')
        continue
    print(f'\n{'='*60}')
    print(f'DESCRIPTIVE STATS: {name.upper()}')
    print(f'{'='*60}')
    print(numeric_df.describe().T.to_string())
    print()

## 5. Key Tables — Focused Inspection

### 5.1 Products Table

In [ ]:
if 'products' in tables:
    products = tables['products']
    print(f'Products Table: {products.shape[0]} rows × {products.shape[1]} columns')
    print()
    print('--- Category Distribution ---')
    print(products['category'].value_counts())
    print()
    print('--- Demand Category Distribution ---')
    if 'demand_category' in products.columns:
        print(products['demand_category'].value_counts())
    print()
    print('--- Unit Cost Statistics ---')
    print(products['unit_cost'].describe())
    print()
    print('--- Sample Rows (Head) ---')
    print(products.head(3).to_string())

### 5.2 Sales Table

In [ ]:
if 'sales' in tables:
    sales = tables['sales']
    print(f'Sales Table: {sales.shape[0]:,} rows × {sales.shape[1]} columns')
    print()
    print('--- Customer Type Distribution ---')
    if 'customer_type' in sales.columns:
        print(sales['customer_type'].value_counts())
    print()
    print('--- Revenue Statistics ---')
    if 'revenue' in sales.columns:
        print(sales['revenue'].describe())
    print()
    print('--- Season Distribution ---')
    if 'season' in sales.columns:
        print(sales['season'].value_counts())
    print()
    print('--- Profit Margin Statistics ---')
    if 'profit_margin_pct' in sales.columns:
        print(sales['profit_margin_pct'].describe())
    print()
    print('--- Sample Rows (Head) ---')
    display_cols = ['sale_id', 'product_id', 'sale_date', 'quantity', 'unit_price', 'revenue', 'profit', 'profit_margin_pct', 'season']
    display_cols = [c for c in display_cols if c in sales.columns]
    print(sales[display_cols].head(5).to_string(index=False))

### 5.3 Inventory Table

In [ ]:
if 'inventory' in tables:
    inventory = tables['inventory']
    print(f'Inventory Table: {inventory.shape[0]:,} rows × {inventory.shape[1]} columns')
    print()
    if 'stock_status' in inventory.columns:
        print('--- Stock Status Distribution ---')
        print(inventory['stock_status'].value_counts())
        print()
    if 'reorder_required' in inventory.columns:
        print('--- Reorder Required ---')
        print(inventory['reorder_required'].value_counts())
        print()
    if 'inventory_value' in inventory.columns:
        total_val = inventory['inventory_value'].sum()
        print(f'--- Total Inventory Value ---')
        print(f'  ₹{total_val:,.2f}')
    print()
    print('--- Sample Rows (Head) ---')
    display_cols = ['inventory_id', 'product_id', 'quantity', 'location', 'stock_status', 'reorder_required', 'inventory_value']
    display_cols = [c for c in display_cols if c in inventory.columns]
    print(inventory[display_cols].head(5).to_string(index=False))

### 5.4 Purchase Orders Table

In [ ]:
if 'purchase_orders' in tables:
    po = tables['purchase_orders']
    print(f'Purchase Orders Table: {po.shape[0]:,} rows × {po.shape[1]} columns')
    print()
    if 'status' in po.columns:
        print('--- PO Status Distribution ---')
        print(po['status'].value_counts())
        print()
    if 'on_time_delivery' in po.columns:
        print('--- On-Time Delivery ---')
        print(po['on_time_delivery'].value_counts())
        print()
    if 'total_order_value' in po.columns:
        print(f'--- Total Procurement Spend ---')
        print(f'  ₹{po["total_order_value"].sum():,.2f}')

### 5.5 Suppliers Table

In [ ]:
if 'suppliers' in tables:
    suppliers = tables['suppliers']
    print(f'Suppliers Table: {suppliers.shape[0]:,} rows × {suppliers.shape[1]} columns')
    print()
    print('--- Rating Statistics ---')
    print(suppliers['rating'].describe())
    print()
    if 'on_time_rate' in suppliers.columns:
        print('--- On-Time Rate Statistics ---')
        print(suppliers['on_time_rate'].describe())
    print()
    print('--- Sample Rows (Head) ---')
    display_cols = ['supplier_id', 'name', 'lead_time_days', 'rating', 'on_time_rate', 'total_order_value']
    display_cols = [c for c in display_cols if c in suppliers.columns]
    print(suppliers[display_cols].head().to_string(index=False))

## 6. Null Value Summary Across All Tables

In [ ]:
print('=== NULL VALUE AUDIT ===')
print(f'{"Table":<25} {"Column":<35} {"Null Count":>12} {"Null %":>10}')
print('-' * 90)

for name, df in tables.items():
    for col in df.columns:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            null_pct = null_count / len(df) * 100
            print(f'{name:<25} {col:<35} {null_count:>12,} {null_pct:>9.2f}%')

print()
print('✅ Null audit complete')

## 7. Uniqueness & Primary Key Validation

In [ ]:
pk_map = {
    'products': 'product_id',
    'suppliers': 'supplier_id',
    'inventory': 'inventory_id',
    'sales': 'sale_id',
    'purchase_orders': 'po_id',
    'forecasts': 'forecast_id',
    'users': 'user_id',
    'audit_logs': 'log_id'
}

print('=== PRIMARY KEY UNIQUENESS CHECK ===')
all_ok = True
for table_name, pk_col in pk_map.items():
    if table_name not in tables:
        continue
    df = tables[table_name]
    if pk_col not in df.columns:
        print(f'  [{table_name}] WARNING: PK column "{pk_col}" not found')
        continue
    total = len(df)
    unique = df[pk_col].nunique()
    null_count = df[pk_col].isnull().sum()
    status = '✅' if (unique == total and null_count == 0) else '❌'
    if status == '❌':
        all_ok = False
    print(f'  {status} [{table_name}] PK={pk_col}: {unique:,}/{total:,} unique, {null_count} nulls')

print()
if all_ok:
    print('✅ All primary keys are unique and non-null')
else:
    print('⚠️  Some primary key issues detected — review above')

## 8. Engineered Features Verification

In [ ]:
expected_features = {
    'sales': ['revenue', 'profit', 'profit_margin_pct', 'week', 'month', 'quarter', 'year', 'season', 'is_weekend'],
    'inventory': ['inventory_value', 'stock_status', 'reorder_required', 'days_until_stockout'],
    'products': ['avg_daily_demand', 'demand_category'],
    'purchase_orders': ['total_order_value', 'on_time_delivery'],
    'suppliers': ['on_time_rate', 'total_order_value']
}

print('=== ENGINEERED FEATURES VERIFICATION ===')
for table_name, feature_cols in expected_features.items():
    if table_name not in tables:
        print(f'  [{table_name}] NOT FOUND')
        continue
    df = tables[table_name]
    present = [c for c in feature_cols if c in df.columns]
    missing = [c for c in feature_cols if c not in df.columns]
    status = '✅' if not missing else '⚠️'
    print(f'  {status} [{table_name}]: {len(present)}/{len(feature_cols)} features present', end='')
    if missing:
        print(f' — Missing: {missing}')
    else:
        print()

print()
print('✅ Feature verification complete')

## 9. EDA Readiness Checklist

In [ ]:
print('=== EDA READINESS ASSESSMENT ===')
print()

checks = []

# Check 1: All core tables present
core_tables = ['products', 'sales', 'inventory', 'suppliers', 'purchase_orders']
missing_tables = [t for t in core_tables if t not in tables]
checks.append(('All core tables loaded', not missing_tables, f'Missing: {missing_tables}' if missing_tables else ''))

# Check 2: Sales table has enough rows
if 'sales' in tables:
    n_sales = len(tables['sales'])
    checks.append(('Sales table has 100+ rows', n_sales >= 100, f'Only {n_sales} rows found'))

# Check 3: Revenue column exists in sales
if 'sales' in tables:
    checks.append(('Revenue feature exists in sales', 'revenue' in tables['sales'].columns, ''))

# Check 4: Stock status in inventory
if 'inventory' in tables:
    checks.append(('Stock status feature exists', 'stock_status' in tables['inventory'].columns, ''))

# Check 5: Season column in sales
if 'sales' in tables:
    checks.append(('Season column exists in sales', 'season' in tables['sales'].columns, ''))

# Check 6: No fully-null columns in sales
if 'sales' in tables:
    fully_null = [c for c in tables['sales'].columns if tables['sales'][c].isnull().all()]
    checks.append(('No fully-null columns in sales', not fully_null, f'Fully null: {fully_null}' if fully_null else ''))

# Check 7: Supplier table has rating
if 'suppliers' in tables:
    checks.append(('Supplier rating column present', 'rating' in tables['suppliers'].columns, ''))

passed = sum(1 for _, ok, _ in checks if ok)
total = len(checks)

for name, ok, detail in checks:
    status = '✅' if ok else '❌'
    line = f'  {status} {name}'
    if detail:
        line += f' — {detail}'
    print(line)

print()
print(f'EDA READINESS: {passed}/{total} checks passed')
if passed == total:
    print()
    print('🚀 Dataset is fully ready for Phase 4 — Exploratory Data Analysis!')
else:
    print()
    print('⚠️  Some checks failed. Review issues above before proceeding to EDA.')

---

## Summary

This notebook has:
- ✅ Loaded all 7 processed CSV datasets from `data/processed/`
- ✅ Verified schemas, column types, and row counts
- ✅ Confirmed engineered feature columns across all tables
- ✅ Audited null values and primary key uniqueness
- ✅ Confirmed EDA readiness

**Next Step → Day 8:** `02_Sales_Analysis.ipynb` — Sales trends, product performance, seasonal patterns

---

*Generated by Antigravity | Phase 3 Day 7 | 2026-07-07*